In [2]:
# ============================================================
# ONE-CELL (ROBUST + FIXED SELECTION): Cross-sectional ranking model (LightGBM REGRESSION)
# DB SOURCE: Postgres public.prices
# WEEKLY REBALANCE: every N trading days (NON-OVERLAPPING realized returns)
# Sector-neutral baskets (configurable strict/soft) + forced totals 10/10
# OOS: Rank IC + LS gross + turnover + cost-adjusted net Sharpe
# Latest-date: stock BUY/SELL/HOLD + confidence %, sector summary
# ============================================================

import os
import numpy as np
import pandas as pd
import psycopg2
from dotenv import load_dotenv
import warnings
warnings.filterwarnings("ignore")

try:
    import lightgbm as lgb
except ImportError:
    import sys, subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lightgbm"])
    import lightgbm as lgb

from scipy.stats import spearmanr
from math import erf, sqrt

# ----------------------------
# 0) CONFIG
# ----------------------------
load_dotenv()
DB_CONFIG = {
    "dbname":   os.getenv("DB_NAME"),
    "user":     os.getenv("DB_USER"),
    "password": os.getenv("DB_PASSWORD"),
    "host":     os.getenv("DB_HOST"),
    "port":     os.getenv("DB_PORT"),
}

symbols = [
    '2222','2380','2381','2382','4030','2030',
    '2010','2020','2060','2310','2350','1211','1321','1322','2223','2250','2290','2150',
    '3030','3092','3004','3050',
    '1212','2320','2040','4142',
    '4190','4001','2280','2050','4164','4240','4003','4161',
    '4013','4004','4009','4017',
    '1120','1180','1150','1050','1060','1080',
    '7010','7020',
    '2082','2080',
    '4300','4321',
    '7203','7202'
]
symbols = [str(s) for s in symbols]
symbols_tuple = tuple(symbols)

sector_map = {
    '2222':'Energy','2380':'Energy','2381':'Energy','2382':'Energy','4030':'Energy','2030':'Energy',
    '2010':'Materials','2020':'Materials','2060':'Materials','2310':'Materials','2350':'Materials','1211':'Materials',
    '1321':'Materials','1322':'Materials','2223':'Materials','2250':'Materials','2290':'Materials','2150':'Materials',
    '3030':'Cement','3092':'Cement','3004':'Cement','3050':'Cement',
    '1120':'Banks','1180':'Banks','1150':'Banks','1050':'Banks','1060':'Banks','1080':'Banks',
    '7010':'Telecom','7020':'Telecom',
    '2082':'Utilities','2080':'Utilities',
    '4013':'Healthcare','4004':'Healthcare','4009':'Healthcare','4017':'Healthcare',
    '4190':'Consumer','4001':'Consumer','2280':'Consumer','2050':'Consumer','4164':'Consumer','4240':'Consumer','4003':'Consumer','4161':'Consumer',
    '4300':'RealEstate','4321':'RealEstate',
    '7203':'Software','7202':'Software',
    '1212':'CapitalGoods','2320':'CapitalGoods','2040':'CapitalGoods','4142':'CapitalGoods'
}

H                      = 5
SPLIT_DATE             = "2025-01-01"
REBALANCE_EVERY_N_DAYS = 5
K_BUY                  = 10
K_SELL                 = 10
COST_ONE_WAY_BPS       = 50.0
SEED                   = 42
SELECTION_MODE         = "soft"
SOFT_MAX_SECTOR_FRAC   = 0.30
USE_BUFFERING          = True
BUFFER                 = 10
USE_REBALANCE_GATE     = True
GATE_ALPHA             = 2

LGB_PARAMS = dict(
    objective        = "regression",
    n_estimators     = 2000,
    learning_rate    = 0.03,
    num_leaves       = 31,
    min_child_samples= 60,
    subsample        = 0.8,
    colsample_bytree = 0.8,
    reg_lambda       = 1.0,
    random_state     = SEED,
    force_col_wise   = True,
)

# ----------------------------
# Helpers
# ----------------------------
def zscore_cs(frame: pd.DataFrame, cols):
    out = frame.copy()
    def _z(s):
        sd = s.std(ddof=0)
        if sd == 0 or np.isnan(sd):
            return s * 0.0
        return (s - s.mean()) / sd
    out[cols] = out.groupby("date")[cols].transform(_z)
    return out

def z_to_conf(z):
    p = 0.5 * (1 + erf(abs(float(z)) / sqrt(2)))
    return float(min(0.99, max(0.50, p)))

def make_equal_weight_ls(long_syms, short_syms):
    long_syms  = pd.Index(list(long_syms)).astype(str).unique().tolist()
    short_syms = pd.Index(list(short_syms)).astype(str).unique().tolist()
    w = pd.Series(dtype=float)
    if len(long_syms)  > 0:
        w = pd.concat([w, pd.Series( 1.0 / len(long_syms),  index=pd.Index(long_syms,  dtype=str))])
    if len(short_syms) > 0:
        w = pd.concat([w, pd.Series(-1.0 / len(short_syms), index=pd.Index(short_syms, dtype=str))])
    w = w.groupby(level=0).sum()
    w = w[w != 0.0]
    return w

def turnover_one_way(prev_w: pd.Series, new_w: pd.Series) -> float:
    if prev_w is None or len(prev_w) == 0: prev_w = pd.Series(dtype=float)
    if new_w  is None or len(new_w)  == 0: new_w  = pd.Series(dtype=float)
    all_syms = prev_w.index.union(new_w.index)
    prev = prev_w.reindex(all_syms).fillna(0.0)
    new  = new_w.reindex(all_syms).fillna(0.0)
    return 0.5 * float((new - prev).abs().sum())

def sharpe_annualized(series, horizon_bars=5):
    s = pd.Series(series).dropna()
    if len(s) < 5: return np.nan   # was 30, lowered for short periods
    mu = float(s.mean())
    sd = float(s.std(ddof=0))
    if sd == 0 or np.isnan(sd): return np.nan
    return float((mu / sd) * np.sqrt(252 / horizon_bars))

def _soft_sector_neutral(gg, score_col, k_buy, k_sell, sector_col="sector", max_sector_frac=0.30):
    gg = gg[["symbol", sector_col, score_col]].dropna().copy()
    gg["symbol"]   = gg["symbol"].astype(str)
    gg[sector_col] = gg[sector_col].astype(str)
    if len(gg) == 0: return set(), set(), {}

    cap_buy  = max(1, int(np.floor(k_buy  * max_sector_frac)))
    cap_sell = max(1, int(np.floor(k_sell * max_sector_frac)))

    gg_desc = gg.sort_values(score_col, ascending=False).reset_index(drop=True)
    gg_asc  = gg.sort_values(score_col, ascending=True).reset_index(drop=True)

    buy, sell       = set(), set()
    buy_counts, sell_counts = {}, {}

    def can_add(sym, side):
        sec = gg.loc[gg["symbol"] == sym, sector_col].iloc[0]
        return buy_counts.get(sec, 0) < cap_buy if side == "buy" else sell_counts.get(sec, 0) < cap_sell

    def add(sym, side):
        sec = gg.loc[gg["symbol"] == sym, sector_col].iloc[0]
        if side == "buy":  buy.add(sym);  buy_counts[sec]  = buy_counts.get(sec, 0)  + 1
        else:              sell.add(sym); sell_counts[sec] = sell_counts.get(sec, 0) + 1

    for _, r in gg_desc.iterrows():
        if len(buy) >= k_buy: break
        sym = r["symbol"]
        if sym in sell: continue
        if can_add(sym, "buy"): add(sym, "buy")

    for _, r in gg_asc.iterrows():
        if len(sell) >= k_sell: break
        sym = r["symbol"]
        if sym in buy: continue
        if can_add(sym, "sell"): add(sym, "sell")

    if len(buy) < k_buy:
        for _, r in gg_desc.iterrows():
            if len(buy) >= k_buy: break
            sym = r["symbol"]
            if sym not in buy and sym not in sell: add(sym, "buy")

    if len(sell) < k_sell:
        for _, r in gg_asc.iterrows():
            if len(sell) >= k_sell: break
            sym = r["symbol"]
            if sym not in buy and sym not in sell: add(sym, "sell")

    return buy, sell, dict(cap_buy=cap_buy, cap_sell=cap_sell)

def _strict_sector_neutral(gg, score_col, k_buy, k_sell, sector_col="sector"):
    gg = gg[["symbol", sector_col, score_col]].dropna().copy()
    gg["symbol"]   = gg["symbol"].astype(str)
    gg[sector_col] = gg[sector_col].astype(str)
    gg = gg.sort_values(score_col, ascending=False)
    n  = len(gg)
    if n == 0: return set(), set(), {}

    sec_sizes = gg[sector_col].value_counts().to_dict()
    sectors   = list(sec_sizes.keys())
    med       = float(gg[score_col].median())

    def allocate(total_k):
        raw    = {s: (sec_sizes[s] / n) * total_k for s in sectors}
        alloc  = {s: int(np.floor(raw[s])) for s in sectors}
        remaining = total_k - sum(alloc.values())
        if remaining > 0:
            frac = sorted(((s, raw[s] - alloc[s]) for s in sectors), key=lambda x: x[1], reverse=True)
            for i in range(remaining): alloc[frac[i % len(frac)][0]] += 1
        for s in sectors: alloc[s] = min(alloc[s], sec_sizes[s])
        return alloc

    buy_alloc  = allocate(k_buy)
    sell_alloc = allocate(k_sell)
    buy, sell  = set(), set()
    forced_positive_shorts = forced_negative_longs = 0

    for s in sectors:
        sub      = gg[gg[sector_col] == s].copy()
        kb, ks   = int(buy_alloc.get(s, 0)), int(sell_alloc.get(s, 0))
        if len(sub) == 0: continue
        sub_buy  = sub[sub[score_col] >= med]
        sub_sell = sub[sub[score_col] <  med]
        buy_pool  = sub_buy  if len(sub_buy)  >= kb else sub
        sell_pool = sub_sell if len(sub_sell) >= ks else sub
        if kb > 0:
            chosen = set(buy_pool.nlargest(kb, score_col)["symbol"])
            forced_negative_longs += sum(sub.set_index("symbol").loc[list(chosen), score_col] < med)
            buy |= chosen
        if ks > 0:
            chosen = set(sell_pool.nsmallest(ks, score_col)["symbol"])
            forced_positive_shorts += sum(sub.set_index("symbol").loc[list(chosen), score_col] >= med)
            sell |= chosen

    overlap = buy & sell
    if overlap:
        sub = gg.set_index("symbol")
        for sym in list(overlap):
            if float(sub.loc[sym, score_col]) >= med: sell.remove(sym)
            else: buy.remove(sym)

    if len(buy)  < k_buy:
        candidates = gg[~gg["symbol"].isin(buy | sell)]
        buy  |= set(candidates.nlargest(k_buy   - len(buy),  score_col)["symbol"])
    if len(sell) < k_sell:
        candidates = gg[~gg["symbol"].isin(buy | sell)]
        sell |= set(candidates.nsmallest(k_sell  - len(sell), score_col)["symbol"])

    buy  = set(list(buy)[:k_buy])
    sell = set(list(sell)[:k_sell])
    return buy, sell, dict(forced_positive_shorts=int(forced_positive_shorts),
                           forced_negative_longs=int(forced_negative_longs),
                           median_score=float(med))

def pick_portfolio(gday, score_col="pred"):
    if SELECTION_MODE.lower() == "soft":
        return _soft_sector_neutral(gday, score_col, K_BUY, K_SELL,
                                    sector_col="sector", max_sector_frac=SOFT_MAX_SECTOR_FRAC)
    else:
        return _strict_sector_neutral(gday, score_col, K_BUY, K_SELL, sector_col="sector")

# ----------------------------
# 1) LOAD DATA
# ----------------------------
conn = psycopg2.connect(**DB_CONFIG)
conn.autocommit = True

q = f"""
SELECT date, symbol, open, high, low, close, volume, turnover, num_trades
FROM public.prices
WHERE symbol IN {symbols_tuple}
ORDER BY date ASC;
"""
df = pd.read_sql_query(q, conn)
df["date"]   = pd.to_datetime(df["date"])
df["symbol"] = df["symbol"].astype(str)
df = df.sort_values(["symbol", "date"]).reset_index(drop=True)
df["sector"] = df["symbol"].map(sector_map)

print("===== DATA CHECK =====")
print("Rows:", len(df))
print("Symbols:", df["symbol"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

# ----------------------------
# 2) FEATURES + LABEL
# ----------------------------
g = df.groupby("symbol", group_keys=False)

df["ret_1d"]    = g["close"].pct_change()
df["fwd_ret_h"] = g["close"].transform(lambda s: s.shift(-H) / s - 1)
df["mom_1w"]    = g["close"].transform(lambda s: s / s.shift(5)  - 1)
df["mom_1m"]    = g["close"].transform(lambda s: s / s.shift(21) - 1)
df["mom_3m"]    = g["close"].transform(lambda s: s / s.shift(63) - 1)
df["sma_20"]    = g["close"].transform(lambda s: s.rolling(20).mean())
df["sma_60"]    = g["close"].transform(lambda s: s.rolling(60).mean())
df["dist_sma_20"]     = df["close"] / df["sma_20"] - 1
df["dist_sma_60"]     = df["close"] / df["sma_60"] - 1
df["vol_5"]           = g["ret_1d"].transform(lambda s: s.rolling(5).std())
df["vol_20"]          = g["ret_1d"].transform(lambda s: s.rolling(20).std())
df["vol_60"]          = g["ret_1d"].transform(lambda s: s.rolling(60).std())
df["vol_ratio_20_60"] = df["vol_20"] / df["vol_60"]
df["vol_sma_20"]      = g["volume"].transform(lambda s: s.rolling(20).mean())
df["vol_surge"]       = df["volume"] / df["vol_sma_20"] - 1

feature_cols = [
    "ret_1d",
    "mom_1w", "mom_1m", "mom_3m",
    "dist_sma_20", "dist_sma_60",
    "vol_5", "vol_20", "vol_ratio_20_60",
    "vol_surge",
    "turnover", "num_trades",
]

df_feat      = df.dropna(subset=feature_cols).copy()
df_trainable = df_feat.dropna(subset=["fwd_ret_h"]).copy()
df_feat      = zscore_cs(df_feat, feature_cols)
df_trainable = zscore_cs(df_trainable, feature_cols)

print("\n===== FEATURES CHECK =====")
print("Rows with features:", len(df_feat))
print("Rows trainable (features + fwd):", len(df_trainable))
print("Latest date in DB:", df["date"].max())
print("Latest scorable date:", df_feat["date"].max())
print("Latest trainable date:", df_trainable["date"].max(), "(~H trading bars before latest)")


===== DATA CHECK =====
Rows: 62576
Symbols: 52
Date range: 2021-01-03 00:00:00 to 2026-04-09 00:00:00

===== FEATURES CHECK =====
Rows with features: 59300
Rows trainable (features + fwd): 59040
Latest date in DB: 2026-04-09 00:00:00
Latest scorable date: 2026-04-09 00:00:00
Latest trainable date: 2026-04-02 00:00:00 (~H trading bars before latest)


In [3]:
# ----------------------------
# 3) TRAIN LightGBM — ROLLING QUARTERLY EXPANDING WINDOW (no leakage)
# ----------------------------

test_all = df_trainable[df_trainable["date"] >= SPLIT_DATE].copy()

print(f"Base train rows (pre-{SPLIT_DATE}): {(df_trainable['date'] < SPLIT_DATE).sum():,}")
print(f"Total test rows: {len(test_all):,}")

last_test_date = pd.Timestamp(test_all["date"].max())

retrain_dates = pd.date_range(
    start  = SPLIT_DATE,
    end    = last_test_date + pd.offsets.MonthEnd(3),
    freq   = "QS"
)

preds_all = []

for i, retrain_date in enumerate(retrain_dates):
    next_date = (retrain_dates[i + 1] if i + 1 < len(retrain_dates)
                 else last_test_date + pd.Timedelta(days=1))

    mask  = df_trainable["date"] < retrain_date
    X_fit = df_trainable[mask][feature_cols]
    y_fit = df_trainable[mask]["fwd_ret_h"].astype(float)

    if len(X_fit) < 500:
        print(f"  Skipping {retrain_date.date()} — only {len(X_fit)} rows")
        continue

    m = lgb.LGBMRegressor(**{**LGB_PARAMS, "n_estimators": 300})
    m.fit(X_fit, y_fit)

    fwd_mask = (test_all["date"] >= retrain_date) & (test_all["date"] < next_date)
    chunk    = test_all[fwd_mask].copy()

    if len(chunk) == 0:
        print(f"  No test rows for {retrain_date.date()} → {next_date.date()}")
        continue

    chunk["pred"] = m.predict(chunk[feature_cols])
    preds_all.append(chunk)
    print(f"  [{retrain_date.date()} → {next_date.date()}]"
          f"  train_n={len(X_fit):,}  scored={len(chunk):,} rows")

test = pd.concat(preds_all).sort_values(["date", "symbol"]).reset_index(drop=True)

print(f"\nTotal scored rows: {len(test):,}")
print(f"Date range: {test['date'].min().date()} → {test['date'].max().date()}")

# ----------------------------
# IC metrics on rolling predictions
# ----------------------------
print("\n===== OOS (rolling quarterly) RANK METRICS =====")

ic_vals, ic_dates = [], []
for d, gday in test.groupby("date", sort=True):
    if gday["pred"].nunique() < 3: continue
    corr = spearmanr(gday["pred"], gday["fwd_ret_h"]).correlation
    if corr == corr:
        ic_dates.append(d)
        ic_vals.append(corr)

ic      = pd.Series(ic_vals, index=pd.to_datetime(ic_dates)).sort_index()
ic_mean = float(ic.mean())
ic_std  = float(ic.std(ddof=0))
ic_t    = ic_mean / (ic_std / np.sqrt(len(ic))) if (ic_std != 0 and len(ic) > 2) else np.nan

print("Daily Spearman IC mean:", round(ic_mean, 4))
print("IC t-stat:", round(ic_t, 3) if ic_t == ic_t else ic_t)
print("IC observations:", int(len(ic)))

Base train rows (pre-2025-01-01): 42,925
Total test rows: 16,115
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 42925, number of used features: 12
[LightGBM] [Info] Start training from score 0.001679
  [2025-01-01 → 2025-04-01]  train_n=42,925  scored=3,172 rows
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 46097, number of used features: 12
[LightGBM] [Info] Start training from score 0.001120
  [2025-04-01 → 2025-07-01]  train_n=46,097  scored=3,011 rows
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 49108, number of used features: 12
[LightGBM] [Info] Start training from score 0.001206
  [2025-07-01 → 2025-10-01]  train_n=49,108  scored=3,380 rows
[LightGBM] [Info] Total Bins 3060
[LightGBM] [Info] Number of data points in the train set: 52488, number of used features: 12
[LightGBM] [Info] Start training from score 0.001307
  [2025-10-01 → 2026-01-01

In [12]:
# At the very end of cell 2, after the loop finishes
import joblib, json
from datetime import datetime

os.makedirs("./../models", exist_ok=True)

joblib.dump(m, "./../models/lgbm_rolling_quarterly.pkl")

meta = {
    "saved_at":          datetime.utcnow().isoformat(),
    "features":          feature_cols,
    "n_estimators":      300,
    "H":                 H,
    "latest_train_date": str(df_trainable["date"].max().date()),
    "latest_data_date":  str(df["date"].max().date()),
    "sector_map":        sector_map,
    "symbols":           symbols,
    "K_BUY":             K_BUY,
    "K_SELL":            K_SELL,
    "SOFT_MAX_SECTOR_FRAC": SOFT_MAX_SECTOR_FRAC,
}
with open("./models/lgbm_rolling_quarterly_meta.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"Model saved → models/lgbm_rolling_quarterly.pkl")
print(f"Trained on {len(X_fit):,} rows  |  latest data: {df['date'].max().date()}")

Model saved → models/lgbm_rolling_quarterly.pkl
Trained on 58,936 rows  |  latest data: 2026-04-09


# Financial Performance Rsults 

In [5]:
# ----------------------------
# 4) OOS weekly rebalance (NON-OVERLAPPING) + EWM smoothing + turnover + costs
# ----------------------------
USE_HYSTERESIS_BUFFER  = True
ENTER_LONG_RANK        = K_BUY
ENTER_SHORT_RANK       = K_SELL
EXIT_LONG_RANK         = K_BUY  + 25
EXIT_SHORT_RANK        = K_SELL + 25
USE_REBALANCE_GATE     = True
GATE_ALPHA             = 2.0
GATE_MIN_DELTA_FRAC    = 0.02
MIN_TURNOVER_TO_CHARGE = 0.0

# ----------------------------
# EWM prediction smoothing
# Rank-normalise first so quarterly retrain scale jumps don't distort smoothing
# halflife=5 = one trading week of memory
# ----------------------------
test["pred_rank"] = test.groupby("date")["pred"].rank(pct=True)

pred_pivot  = test.pivot(index="date", columns="symbol", values="pred_rank")
pred_smooth = pred_pivot.apply(lambda col: col.ewm(halflife=5).mean())
pred_long   = pred_smooth.stack().rename("pred_smooth").reset_index()
test        = test.merge(pred_long, on=["date", "symbol"], how="left")
test["pred"] = test["pred_smooth"].fillna(test["pred_rank"])

print("EWM smoothing applied  (halflife=5 days, rank-normalised)")

# ----------------------------
# Rebalance loop
# ----------------------------
cost_one_way    = COST_ONE_WAY_BPS / 10000.0
test_dates      = pd.Index(sorted(test["date"].unique()))
rebalance_dates = [d for i, d in enumerate(test_dates) if i % REBALANCE_EVERY_N_DAYS == 0]

prev_w                  = pd.Series(dtype=float)
prev_longs, prev_shorts = set(), set()
rows                    = []
first_debug             = False

def _rank_frame(gday, score_col="pred"):
    r = gday[["symbol", "sector", score_col]].dropna().copy()
    r["symbol"] = r["symbol"].astype(str)
    r = r.sort_values(score_col, ascending=False).reset_index(drop=True)
    r["rank"]             = np.arange(1, len(r) + 1)
    r["rank_from_bottom"] = np.arange(len(r), 0, -1)
    return r

def _soft_pick_with_caps(r, k_buy, k_sell, sector_col="sector", max_sector_frac=0.30):
    cap_buy  = max(1, int(np.floor(k_buy  * max_sector_frac)))
    cap_sell = max(1, int(np.floor(k_sell * max_sector_frac)))
    buy, sell       = [], []
    buy_ct, sell_ct = {}, {}

    for _, row in r.iterrows():
        if len(buy) >= k_buy: break
        sym, sec = row["symbol"], row[sector_col]
        if sym in sell: continue
        if buy_ct.get(sec, 0) < cap_buy:
            buy.append(sym); buy_ct[sec] = buy_ct.get(sec, 0) + 1

    for _, row in r.iloc[::-1].iterrows():
        if len(sell) >= k_sell: break
        sym, sec = row["symbol"], row[sector_col]
        if sym in buy: continue
        if sell_ct.get(sec, 0) < cap_sell:
            sell.append(sym); sell_ct[sec] = sell_ct.get(sec, 0) + 1

    if len(buy) < k_buy:
        for sym in r["symbol"].tolist():
            if len(buy) >= k_buy: break
            if sym not in buy and sym not in sell: buy.append(sym)

    if len(sell) < k_sell:
        for sym in r["symbol"].tolist()[::-1]:
            if len(sell) >= k_sell: break
            if sym not in buy and sym not in sell: sell.append(sym)

    return set(buy[:k_buy]), set(sell[:k_sell])

def _hysteresis_update(r, prev_longs, prev_shorts):
    rank_map = r.set_index("symbol")[["rank", "rank_from_bottom"]]

    keep_longs  = {sym for sym in prev_longs
                   if sym in rank_map.index
                   and int(rank_map.loc[sym, "rank"]) <= EXIT_LONG_RANK}
    keep_shorts = {sym for sym in prev_shorts
                   if sym in rank_map.index
                   and int(rank_map.loc[sym, "rank_from_bottom"]) <= EXIT_SHORT_RANK}
    keep_longs  -= keep_shorts
    keep_shorts -= keep_longs
    longs, shorts = set(keep_longs), set(keep_shorts)

    for sym in r[r["rank"] <= ENTER_LONG_RANK]["symbol"].tolist():
        if len(longs) >= K_BUY: break
        if sym not in shorts: longs.add(sym)

    for sym in r[r["rank_from_bottom"] <= ENTER_SHORT_RANK]["symbol"].tolist():
        if len(shorts) >= K_SELL: break
        if sym not in longs: shorts.add(sym)

    if len(longs) < K_BUY:
        for sym in r["symbol"].tolist():
            if len(longs) >= K_BUY: break
            if sym not in longs and sym not in shorts: longs.add(sym)

    if len(shorts) < K_SELL:
        for sym in r["symbol"].tolist()[::-1]:
            if len(shorts) >= K_SELL: break
            if sym not in longs and sym not in shorts: shorts.add(sym)

    return set(list(longs)[:K_BUY]), set(list(shorts)[:K_SELL])

for d in rebalance_dates:
    gday = test.loc[test["date"] == d].copy()
    r    = _rank_frame(gday, score_col="pred")

    base_longs, base_shorts = _soft_pick_with_caps(
        r, K_BUY, K_SELL, sector_col="sector", max_sector_frac=SOFT_MAX_SECTOR_FRAC
    )

    if USE_HYSTERESIS_BUFFER:
        cand_longs, cand_shorts = _hysteresis_update(r, prev_longs, prev_shorts)
        selector_diag = {
            "mode":             "soft+hysteresis",
            "enter_long_rank":  ENTER_LONG_RANK,
            "exit_long_rank":   EXIT_LONG_RANK,
            "enter_short_rank": ENTER_SHORT_RANK,
            "exit_short_rank":  EXIT_SHORT_RANK,
            "cap_buy":          int(np.floor(K_BUY  * SOFT_MAX_SECTOR_FRAC)),
            "cap_sell":         int(np.floor(K_SELL * SOFT_MAX_SECTOR_FRAC)),
        }
    else:
        cand_longs, cand_shorts = base_longs, base_shorts
        selector_diag = {
            "mode":     "soft_only",
            "cap_buy":  int(np.floor(K_BUY  * SOFT_MAX_SECTOR_FRAC)),
            "cap_sell": int(np.floor(K_SELL * SOFT_MAX_SECTOR_FRAC)),
        }

    cand_w   = make_equal_weight_ls(cand_longs, cand_shorts)
    
    to       = turnover_one_way(prev_w, cand_w)
    est_cost = 0.0 if to <= MIN_TURNOVER_TO_CHARGE else to * cost_one_way

    traded                = True
    delta_frac, cost_frac = 0.0, 0.0

    if USE_REBALANCE_GATE:
        s       = gday.set_index("symbol")["pred"]
        exp_new = float((cand_w * s.reindex(cand_w.index)).fillna(0.0).sum())
        exp_old = (0.0 if (prev_w is None or len(prev_w) == 0)
                   else float((prev_w * s.reindex(prev_w.index)).fillna(0.0).sum()))
        delta       = exp_new - exp_old
        pred_spread = float(gday["pred"].quantile(0.90) - gday["pred"].quantile(0.10))

        if pred_spread > 0:
            delta_frac = delta    / pred_spread
            cost_frac  = est_cost / pred_spread
        else:
            delta_frac = cost_frac = 0.0

        if delta_frac <= GATE_ALPHA * cost_frac or delta_frac <= GATE_MIN_DELTA_FRAC:
            cand_w                  = prev_w
            cand_longs, cand_shorts = prev_longs, prev_shorts
            to       = 0.0
            est_cost = 0.0
            traded   = False

    rets  = gday.set_index("symbol")["fwd_ret_h"]
    gross = float((cand_w * rets.reindex(cand_w.index)).fillna(0.0).sum())
    net   = gross - est_cost

    if not first_debug:
        print("\n===== WEEKLY REBALANCE DEBUG (first rebalance date) =====")
        print("Date:", pd.to_datetime(d).date())
        print("Selector diag:", selector_diag)
        print("Gate:", USE_REBALANCE_GATE, "alpha:", GATE_ALPHA,
              "| min_delta_frac:", GATE_MIN_DELTA_FRAC)
        print("Traded?:", traded)
        print(f"  delta_frac={round(delta_frac,4)}"
              f"  threshold_cost={round(GATE_ALPHA*cost_frac,4)}"
              f"  floor={GATE_MIN_DELTA_FRAC}")
        print("Turnover one-way:", round(to, 4), f"(cost={round(est_cost*1e4,2)} bps)")
        first_debug = True

    rows.append({
        "date":             d,
        "traded":           traded,
        "gross_ls":         gross,
        "net_ls":           net,
        "turnover_one_way": to,
        "cost":             est_cost,
    })

    prev_w                  = cand_w
    prev_longs, prev_shorts = (
        set(cand_w[cand_w > 0].index),
        set(cand_w[cand_w < 0].index)
    )

bt_weekly = pd.DataFrame(rows).set_index("date").sort_index()

print("\n===== OOS LONG-SHORT - WEEKLY (NON-OVERLAPPING) WITH TURNOVER CONTROL =====")
print("Weekly trades:", len(bt_weekly), "| Actually traded:", int(bt_weekly["traded"].sum()))
print(f"Avg gross fwd_{H}d LS:", round(bt_weekly["gross_ls"].mean(), 6))
print(f"Avg net   fwd_{H}d LS:", round(bt_weekly["net_ls"].mean(), 6))
print("Avg turnover (one-way):", round(bt_weekly["turnover_one_way"].mean(), 3))
print("Gross ann. Sharpe (approx):",
      round(sharpe_annualized(bt_weekly["gross_ls"], horizon_bars=H), 4))
print("Net   ann. Sharpe (approx):",
      round(sharpe_annualized(bt_weekly["net_ls"],   horizon_bars=H), 4))
print("\nTurnover one-way quantiles:")
print(bt_weekly["turnover_one_way"].quantile([0, 0.25, 0.5, 0.75, 0.9, 0.95, 1.0]).round(3))

EWM smoothing applied  (halflife=5 days, rank-normalised)

===== WEEKLY REBALANCE DEBUG (first rebalance date) =====
Date: 2025-01-01
Selector diag: {'mode': 'soft+hysteresis', 'enter_long_rank': 10, 'exit_long_rank': 35, 'enter_short_rank': 10, 'exit_short_rank': 35, 'cap_buy': 3, 'cap_sell': 3}
Gate: True alpha: 2.0 | min_delta_frac: 0.02
Traded?: True
  delta_frac=1.0294  threshold_cost=0.0127  floor=0.02
Turnover one-way: 1.0 (cost=50.0 bps)

===== OOS LONG-SHORT - WEEKLY (NON-OVERLAPPING) WITH TURNOVER CONTROL =====
Weekly trades: 63 | Actually traded: 46
Avg gross fwd_5d LS: 0.00318
Avg net   fwd_5d LS: 0.002473
Avg turnover (one-way): 0.141
Gross ann. Sharpe (approx): 1.1498
Net   ann. Sharpe (approx): 0.9027

Turnover one-way quantiles:
0.00    0.0
0.25    0.0
0.50    0.1
0.75    0.2
0.90    0.3
0.95    0.3
1.00    1.0
Name: turnover_one_way, dtype: float64


In [6]:
#----- Cell 5 -----
first_half  = bt_weekly[bt_weekly.index <  "2026-01-01"]
second_half = bt_weekly[bt_weekly.index >= "2026-01-01"]

for label, half in [("H1 2025", first_half), ("H2 2026", second_half)]:
    g  = sharpe_annualized(half["gross_ls"], horizon_bars=H)
    n  = sharpe_annualized(half["net_ls"],   horizon_bars=H)
    wr = (half["gross_ls"] > 0).mean()
    print(f"{label}: gross={round(g,3)}  net={round(n,3)}  n={len(half)}  win_rate={round(wr,3)}")

H1 2025: gross=1.151  net=0.904  n=51  win_rate=0.549
H2 2026: gross=1.149  net=0.903  n=12  win_rate=0.667


In [7]:
skipped = bt_weekly[~bt_weekly["traded"]]
print(skipped[["gross_ls", "net_ls", "turnover_one_way"]])

            gross_ls    net_ls  turnover_one_way
date                                            
2025-01-22 -0.008271 -0.008271               0.0
2025-02-05 -0.016337 -0.016337               0.0
2025-02-19 -0.015390 -0.015390               0.0
2025-05-28  0.009210  0.009210               0.0
2025-06-04 -0.015036 -0.015036               0.0
2025-06-24 -0.008182 -0.008182               0.0
2025-07-15  0.009525  0.009525               0.0
2025-09-09 -0.010531 -0.010531               0.0
2025-09-24  0.005576  0.005576               0.0
2025-10-08  0.033710  0.033710               0.0
2025-10-22 -0.002291 -0.002291               0.0
2025-10-29 -0.025260 -0.025260               0.0
2025-11-05  0.003305  0.003305               0.0
2025-12-17  0.045003  0.045003               0.0
2025-12-31 -0.002603 -0.002603               0.0
2026-01-07  0.014850  0.014850               0.0
2026-02-26 -0.048790 -0.048790               0.0


In [8]:
cum         = (1 + bt_weekly["net_ls"]).cumprod()
rolling_max = cum.cummax()
drawdown    = (cum / rolling_max) - 1

ann_ret = bt_weekly["net_ls"].mean() * (252 / H)
calmar  = ann_ret / abs(drawdown.min())

print("Max drawdown:      ", round(drawdown.min(), 4))
print("Max drawdown date: ", drawdown.idxmin().date())
print("Ann. net return:   ", round(ann_ret, 4))
print("Calmar ratio:      ", round(calmar, 2))

Max drawdown:       -0.0776
Max drawdown date:  2026-03-12
Ann. net return:    0.1247
Calmar ratio:       1.61


In [9]:
#---- Cell 8 ----
mkt_weekly = (
    test.groupby("date")["fwd_ret_h"]
    .mean()
    .reindex(bt_weekly.index)
)

first_half  = bt_weekly[bt_weekly.index <  "2026-01-01"]
second_half = bt_weekly[bt_weekly.index >= "2026-01-01"]

print("── Market correlation check ──────────────────────────────")
for label, half in [("H1 2025", first_half), ("H2 2026", second_half)]:
    mkt  = mkt_weekly.reindex(half.index).dropna()
    ls   = half["gross_ls"].reindex(mkt.index)
    corr = ls.corr(mkt)
    beta = ls.cov(mkt) / mkt.var()
    print(f"{label}:  corr={round(corr,3)}  beta={round(beta,3)}  n={len(mkt)}")

mkt_full = mkt_weekly.reindex(bt_weekly.index).dropna()
ls_full  = bt_weekly["gross_ls"].reindex(mkt_full.index)
print(f"\nFull OOS:  corr={round(ls_full.corr(mkt_full),3)}  "
      f"beta={round(ls_full.cov(mkt_full)/mkt_full.var(),3)}")

── Market correlation check ──────────────────────────────
H1 2025:  corr=0.396  beta=0.318  n=51
H2 2026:  corr=0.236  beta=0.202  n=12

Full OOS:  corr=0.359  beta=0.289
